In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
path = "../data/raw/Telco_customer_churn.xlsx"

In [3]:
DROP_COLUMNS = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Lat Long",
    "Churn Reason"
]
def load_data(path):
    df = pd.read_excel(path)
    df = df.drop(columns=DROP_COLUMNS)
    return df

In [4]:
df = load_data(path)

In [5]:
yes_no_cols = [
    "Partner",
    "Dependents",
    "Phone Service",
    "Paperless Billing",
    'Senior Citizen',
]

df['Gender'] = df['Gender'].map({
    'Male' :1,
    'Female' :0 
})
df['Churn Label'] = df['Churn Label'].map({
    'Yes' :1,
    'No' :0 
})

for col in yes_no_cols:
    df[col] = df[col].map({
        'Yes': 1,
        'No': 0 
    })

In [6]:
categorical_cols = [
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Payment Method"
]
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [7]:
df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Tech Support_Yes,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,Los Angeles,90003,33.964131,-118.272783,1,0,0,0,2,1,...,0,0,0,0,0,0,0,0,0,1
1,Los Angeles,90005,34.059281,-118.307420,0,0,0,1,2,1,...,0,0,0,0,0,0,0,0,1,0
2,Los Angeles,90006,34.048013,-118.293953,0,0,0,1,8,1,...,0,0,1,0,1,0,0,0,1,0
3,Los Angeles,90010,34.062125,-118.315709,0,0,1,1,28,1,...,1,0,1,0,1,0,0,0,1,0
4,Los Angeles,90015,34.039224,-118.266293,1,0,0,1,49,1,...,0,0,1,0,1,0,0,0,0,0


In [8]:
df.columns

Index(['City', 'Zip Code', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Churn Label',
       'Churn Value', 'Churn Score', 'CLTV', 'Multiple Lines_No phone service',
       'Multiple Lines_Yes', 'Internet Service_Fiber optic',
       'Internet Service_No', 'Online Security_No internet service',
       'Online Security_Yes', 'Online Backup_No internet service',
       'Online Backup_Yes', 'Device Protection_No internet service',
       'Device Protection_Yes', 'Tech Support_No internet service',
       'Tech Support_Yes', 'Streaming TV_No internet service',
       'Streaming TV_Yes', 'Streaming Movies_No internet service',
       'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year',
       'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check'],
      dtype='str')

In [9]:
drop_cols = [
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
    "Churn Value",
    "Churn Score"
]

df.drop(columns=drop_cols, inplace=True)

In [10]:
df.columns

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Paperless Billing', 'Monthly Charges',
       'Total Charges', 'Churn Label', 'CLTV',
       'Multiple Lines_No phone service', 'Multiple Lines_Yes',
       'Internet Service_Fiber optic', 'Internet Service_No',
       'Online Security_No internet service', 'Online Security_Yes',
       'Online Backup_No internet service', 'Online Backup_Yes',
       'Device Protection_No internet service', 'Device Protection_Yes',
       'Tech Support_No internet service', 'Tech Support_Yes',
       'Streaming TV_No internet service', 'Streaming TV_Yes',
       'Streaming Movies_No internet service', 'Streaming Movies_Yes',
       'Contract_One year', 'Contract_Two year',
       'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check'],
      dtype='str')

In [11]:
service_cols = [
    "Online Security_Yes",
    "Online Backup_Yes",
    "Device Protection_Yes",
    "Tech Support_Yes",
    "Streaming TV_Yes",
    "Streaming Movies_Yes"
]

df['Total Services'] = df[service_cols].sum(axis=1)

In [12]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

In [13]:
df['Average Spend'] = df["Total Charges"] / (df["Tenure Months"] + 1)

In [14]:
median = df["Monthly Charges"].median()

df["High Value"] = (
    df["Monthly Charges"] > median
).astype(int)

In [15]:
def tenure_group(tenure):
    if tenure <= 12:
        return "New"
    elif tenure <= 36:
        return "Regular"
    elif tenure <= 60:
        return "Loyal"
    else:
        return "Veteran"


df["Tenure Group"] = df["Tenure Months"].apply(tenure_group)

In [16]:
engagement_cols = [
    "Partner",
    "Dependents",
    "Online Security_Yes",
    "Online Backup_Yes",
    "Device Protection_Yes",
    "Tech Support_Yes"
]

df["Engagement Score"] = df[engagement_cols].sum(axis=1)

In [17]:
df["HighCharge_MonthlyContract"] = (
    (df["Monthly Charges"] > df["Monthly Charges"].median()) &
    (df["Contract_One year"] == 0) &
    (df["Contract_Two year"] == 0)
).astype(int)

In [18]:
df["Has Internet"] = (
    1 - df["Internet Service_No"]
)

In [19]:
df.columns

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Paperless Billing', 'Monthly Charges',
       'Total Charges', 'Churn Label', 'CLTV',
       'Multiple Lines_No phone service', 'Multiple Lines_Yes',
       'Internet Service_Fiber optic', 'Internet Service_No',
       'Online Security_No internet service', 'Online Security_Yes',
       'Online Backup_No internet service', 'Online Backup_Yes',
       'Device Protection_No internet service', 'Device Protection_Yes',
       'Tech Support_No internet service', 'Tech Support_Yes',
       'Streaming TV_No internet service', 'Streaming TV_Yes',
       'Streaming Movies_No internet service', 'Streaming Movies_Yes',
       'Contract_One year', 'Contract_Two year',
       'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check',
       'Total Services', 'Average Spend', 'High Value', 'Tenure Group',
       'Engagement Score', 'HighCharge_

In [20]:
df = pd.get_dummies(
    df,
    columns=["Tenure Group"],
    drop_first=True,
    dtype=int
)

In [21]:
df.isnull().sum()
df.dropna(inplace=True)

In [22]:
drop_cols = [
    # Engineered Features
    "Total Services",
    "Engagement Score",
    "Has Internet",

    # Redundant "No Internet" dummy variables
    "Online Security_No internet service",
    "Online Backup_No internet service",
    "Device Protection_No internet service",
    "Tech Support_No internet service",
    "Streaming TV_No internet service",
    "Streaming Movies_No internet service",

    # Redundant because Phone Service already tells us this
    "Multiple Lines_No phone service"
]

df.drop(columns=drop_cols, inplace=True)

In [23]:
drop_cols2 = [
    "High Value",
    "Average Spend",
    "Tenure Group_New",
    "Tenure Group_Regular",
    "Tenure Group_Veteran"
]

df.drop(columns=drop_cols2, inplace=True)


In [24]:
df.drop("Phone Service" , axis =1, inplace=True)

In [25]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn Label"])
y = df['Churn Label']

In [26]:
X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y 
)

In [27]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif = pd.DataFrame()
vif["Feature"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif.sort_values("VIF", ascending=False)

,Feature,VIF
6,Monthly Charges,63.974605
7,Total Charges,25.417108
4,Tenure Months,22.850054
8,CLTV,15.482651
10,Internet Service_Fiber optic,13.135573
23,HighCharge_MonthlyContract,5.411822
19,Contract_Two year,4.330270
17,Streaming Movies_Yes,3.404496
16,Streaming TV_Yes,3.375641
11,Internet Service_No,3.257545


In [28]:
df.to_csv('../data/processed/processed_data.csv', index=False)

In [31]:
df.columns


Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Churn Label',
       'CLTV', 'Multiple Lines_Yes', 'Internet Service_Fiber optic',
       'Internet Service_No', 'Online Security_Yes', 'Online Backup_Yes',
       'Device Protection_Yes', 'Tech Support_Yes', 'Streaming TV_Yes',
       'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year',
       'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check',
       'HighCharge_MonthlyContract'],
      dtype='str')